# 05 — Model Training: attorney & litigation on note embeddings

1. Build each claim's running history from its note embeddings: 8 element-wise summaries (mean, max, min, sum,
   max÷min, max−min, first÷last, first−last), each a 1,536-dim vector.
2. Select rows at the day-grid cutoffs per coverage, reusing `02`'s snapshots (same rows, targets and split).
3. Train the network (own encoder per summary → fusion → MLP head), tuned with Optuna on validation Gini.
4. Evaluate on train / val / holdout (coverage-level snapshot rows), and Gini by knowledge-cutoff day.

In [ ]:
import io, json, os, time
import numpy as np
import pandas as pd
import boto3
import torch
import torch.nn as nn

import sys; sys.path.insert(0, '..')   # repo root, so config.py can be imported
from config import S3_BUCKET, EMBEDDING_PREFIX, SNAPSHOT_PREFIX   # paths: config.py
EMB_PREFIX = f'{EMBEDDING_PREFIX}/embeddings'          # written by 04-Embedding-Inference.ipynb
GROUPS = ['mean', 'max', 'min', 'sum', 'max_div_min', 'max_minus_min', 'first_div_last', 'first_minus_last']
EMB_DIM, EPS = 1536, 1e-6
s3 = boto3.client('s3')

## Step 1 — running per-claim features from note embeddings (CPU)

In [2]:
def safe_div(a, b):
    b = np.where(np.abs(b) < EPS, np.sign(b) * EPS + (b == 0) * EPS, b)
    return a / b


def running_features(vecs):
    # vecs: (n_notes, 1536) for one claim, sorted by date -> one row per note, 8 summaries each
    cs = np.cumsum(vecs, axis=0)
    n = np.arange(1, len(vecs) + 1)[:, None]
    mx, mn = np.maximum.accumulate(vecs), np.minimum.accumulate(vecs)
    first = np.broadcast_to(vecs[0], vecs.shape)
    return {'mean': cs / n, 'max': mx, 'min': mn, 'sum': cs,
            'max_div_min': safe_div(mx, mn), 'max_minus_min': mx - mn,
            'first_div_last': safe_div(first, vecs), 'first_minus_last': first - vecs}


emb_rows = []
for k in range(20):
    body = s3.get_object(Bucket=S3_BUCKET, Key=f'{EMB_PREFIX}/notes_{k:03d}.jsonl.out')['Body']  # S3 read
    emb_rows += [json.loads(l) for l in body.iter_lines()]
meta = pd.DataFrame([{k: r[k] for k in ('note_id', 'claim_num', 'date')} for r in emb_rows])
emb = np.array([r['embedding'] for r in emb_rows], dtype=np.float16)
print(f'loaded {len(meta):,} notes, emb {emb.shape} {emb.dtype} ({emb.nbytes / 1e9:.1f} GB)')

meta['seq'] = meta.note_id.str.rsplit('_', n=1).str[1].astype(int)
order = meta.sort_values(['claim_num', 'date', 'seq']).index.to_numpy()
meta, emb = meta.loc[order].reset_index(drop=True), emb[order]

feats = {g: [] for g in GROUPS}
for clm, idx in meta.groupby('claim_num', sort=False).indices.items():
    f = running_features(emb[idx].astype(np.float32))
    for g in GROUPS:
        feats[g].append(f[g].astype(np.float16))
feats = {g: np.concatenate(v) for g, v in feats.items()}
meta['n_notes'] = meta.groupby('claim_num').cumcount() + 1
print(f'DONE: {meta.claim_num.nunique():,} claims, {len(meta):,} rows per feature type x {len(GROUPS)} types')

loaded 4,645,819 notes, emb (4645819, 1536) float16 (14.3 GB)
DONE: 138,233 claims, 4,645,819 rows per feature type x 8 types


## Step 2 — cutoff-based snapshot selection (same rows as `02`)

Instead of one row per note, each model only sees the claim **at its day-grid cutoffs**, exactly like the
LLM-feature datasets in `02`. We reuse `02`'s snapshot keys directly: per coverage, the boundary is the event
date if it happened (else coverage close; severity = last close date), cutoffs are `CLM_OPEN_DT + day` on each
model's grid, unchanged note counts are deduplicated and 0-note snapshots skipped. Each snapshot row then takes
the claim's 8 running summaries as of its `N_NOTES`, so both strategies train on identical rows, targets and splits.

In [3]:
FE_PREFIX = SNAPSHOT_PREFIX      # written by 02-Dataset-Development.ipynb
KEY_COLS = ['CLM_CVG_PK', 'CLM_NUM', 'SNAPSHOT_NUM', 'SNAPSHOT_DAY', 'N_NOTES']
TARGET_COL = {'Attorney': 'TARGET', 'Litigation': 'TARGET', 'Severity': 'TARGET_BUCKET'}

# start row of each claim in the (claim, date)-sorted feature arrays from Step 1
claim_start = meta.reset_index().groupby('claim_num')['index'].min()


def load_snapshot_keys(model):
    frames = []
    for split in ['train', 'val', 'holdout']:
        pref = f'{FE_PREFIX}/{model}/{split}/'
        for page in s3.get_paginator('list_objects_v2').paginate(Bucket=S3_BUCKET, Prefix=pref):  # S3 read
            for o in page.get('Contents', []):
                b = s3.get_object(Bucket=S3_BUCKET, Key=o['Key'])['Body'].read()  # S3 read
                # reads local data
                df = pd.read_parquet(io.BytesIO(b), columns=KEY_COLS + [TARGET_COL[model]])  # no NOTE_TEXT needed
                frames.append(df.assign(split=split))
    return pd.concat(frames, ignore_index=True)


datasets = {}
for model in ['Attorney', 'Litigation', 'Severity']:
    keys = load_snapshot_keys(model)
    keys['feat_row'] = keys.CLM_NUM.map(claim_start) + keys.N_NOTES - 1     # claim's features as of N_NOTES
    assert keys.feat_row.notna().all()
    datasets[model] = keys
    print(f'[{model}] {len(keys):,} snapshot rows')

total = sum(len(d) for d in datasets.values())
print(f'\ntotal rows across all three models: {total:,}  (vs 4,645,819 one-row-per-note)')

[Attorney] 466,090 snapshot rows
[Litigation] 388,956 snapshot rows
[Severity] 619,029 snapshot rows

total rows across all three models: 1,474,075  (vs 4,645,819 one-row-per-note)


In [ ]:
def to_arrays(keys, model):
    # (rows, 8, 1536) float16 view built from the Step-1 feature arrays
    idx = keys.feat_row.to_numpy(dtype=np.int64)
    X = np.stack([feats[g][idx] for g in GROUPS], axis=1)
    y = keys[TARGET_COL[model]].to_numpy()
    return X, y, keys.CLM_CVG_PK.to_numpy(), keys.SNAPSHOT_DAY.to_numpy()

att_data = {s: to_arrays(datasets['Attorney'][datasets['Attorney'].split == s], 'Attorney') for s in ['train', 'val', 'holdout']}
lit_data = {s: to_arrays(datasets['Litigation'][datasets['Litigation'].split == s], 'Litigation') for s in ['train', 'val', 'holdout']}

## Step 3 — the network

In [ ]:
def _block(i, o, p):
    return nn.Sequential(nn.Linear(i, o), nn.LayerNorm(o), nn.GELU(), nn.Dropout(p))


class ClaimNet(nn.Module):
    # one encoder per summary -> concat -> fusion -> MLP head -> 1 logit (same design for both targets)
    def __init__(self, group_dims, fuse_dim=256, head_dims=(128,), in_dim=EMB_DIM, dropout=0.1):
        super().__init__()
        self.encoders = nn.ModuleList([_block(in_dim, d, dropout) for d in group_dims])
        self.fuse = _block(sum(group_dims), fuse_dim, dropout)
        layers, prev = [], fuse_dim
        for h in head_dims:
            layers.append(_block(prev, h, dropout)); prev = h
        self.head = nn.Sequential(*layers, nn.Linear(prev, 1))

    def forward(self, x):                      # x: (B, 8, 1536)
        z = torch.cat([enc(x[:, i]) for i, enc in enumerate(self.encoders)], dim=1)
        return self.head(self.fuse(z)).squeeze(-1)

## Step 4 — Optuna tuning (SageMaker GPU training job)

Search space: lr 1e-4–5e-3 (log), dropout 0–0.7, weight decay 1e-6–3e-2 (log), feature-noise 0–0.05,
batch {32 … 32,768}, fusion dim {128, 256, 384, 512}, per-summary dim {64, 96, 128, 192, 256}.
Fixed: BCEWithLogits with `pos_weight` = neg/pos, AdamW, cosine LR, max 60 epochs, early stop after 8 epochs
without validation-Gini gain. 50 trials per target (43 attorney and 49 litigation completed), best picked by validation Gini.

In [ ]:
from sklearn.metrics import roc_auc_score
gini = lambda y, p: 2 * roc_auc_score(y, p) - 1


def sample_cfg(trial):
    return {'lr': trial.suggest_float('lr', 1e-4, 5e-3, log=True),
            'dropout': trial.suggest_float('dropout', 0.0, 0.7),
            'weight_decay': trial.suggest_float('weight_decay', 1e-6, 3e-2, log=True),
            'noise_std': trial.suggest_float('noise_std', 0.0, 0.05),
            'batch': trial.suggest_categorical('batch', [32, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768]),
            'fuse_dim': trial.suggest_categorical('fuse_dim', [128, 256, 384, 512]),
            'group_dims': [trial.suggest_categorical(f'dim_{g}', [64, 96, 128, 192, 256]) for g in GROUPS]}


def train_one(cfg, Xtr, ytr, Xva, yva, max_epochs=60, patience=8, device='cuda'):
    net = ClaimNet(cfg['group_dims'], cfg['fuse_dim'], dropout=cfg['dropout']).to(device)
    pos = float(ytr.sum()); crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(len(ytr) - pos) / pos], device=device))
    opt = torch.optim.AdamW(net.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    best, best_state, since = -1, None, 0
    for ep in range(max_epochs):
        net.train()
        for i in torch.randperm(len(ytr)).split(cfg['batch']):
            xb = torch.as_tensor(Xtr[i.numpy()], device=device).float()
            xb = xb + cfg['noise_std'] * torch.randn_like(xb)
            loss = crit(net(xb), torch.as_tensor(ytr[i.numpy()], device=device).float())
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        g = gini(yva, predict(net, Xva, device))
        if g > best:
            best, best_state, since = g, {k: v.clone() for k, v in net.state_dict().items()}, 0
        else:
            since += 1
            if since >= patience:
                break
    net.load_state_dict(best_state)
    return net, best


@torch.no_grad()
def predict(net, X, device='cuda', bs=32768):
    net.eval()
    return np.concatenate([torch.sigmoid(net(torch.as_tensor(X[i:i + bs], device=device).float())).cpu().numpy()
                           for i in range(0, len(X), bs)])

### Step 4a — launch the tuning job (one per target)

The search runs as a SageMaker training job on one `ml.g5.4xlarge`. Code is in `tuning/`:
- `claimnet.py`: the network, training loop and search space above (shared with the job)
- `train_optuna.py`: job entry point; runs the trials, picks the best on **validation Gini**, writes
  `runs/<study>/all_trials.json` and `_SUMMARY.json` to S3 (read back in Step 4b)
- `launch_tuning.py`: uploads the train / val / holdout arrays and starts the job

In [ ]:
from tuning.launch_tuning import upload_arrays, launch

# about 12 h per target; runs in the background (wait=False), results land in RUN_PREFIX/<study>/
att_job = launch('Attorney', 'attorney', upload_arrays(att_data, 'Attorney'))
lit_job = launch('Litigation', 'litigation', upload_arrays(lit_data, 'Litigation'))

### Step 4b — every trial and its score (from the saved study files)

In [7]:
RUN_PREFIX = f'{EMBEDDING_PREFIX}/runs'
STUDIES = {'Attorney': 'attorney', 'Litigation': 'litigation'}
GROUPS_SHORT = ['mean', 'max', 'min', 'sum', 'max_div_min', 'max_minus_min', 'first_div_last', 'first_minus_last']


def load_json(key):
    return json.loads(s3.get_object(Bucket=S3_BUCKET, Key=key)['Body'].read())  # S3 read


def show_trials(name, top=10):
    study = STUDIES[name]
    summ = load_json(f'{RUN_PREFIX}/{study}/_SUMMARY.json')  # S3 read
    trials = pd.DataFrame([{'trial': t['trial'], 'val_gini': t['val_gini'], 'holdout_gini': t['holdout_gini'],
                            'val_last_note': t['claim_gini']['val'], 'best_epoch': t['best_epoch'],
                            'lr': t['params']['lr'], 'dropout': t['params']['dropout'],
                            'batch': t['params']['batch'], 'fuse_dim': t['params']['fuse_dim']}
                           for t in load_json(f'{RUN_PREFIX}/{study}/all_trials.json')])  # S3 read
    print(f"{name} ({study}): {summ['n_trials']} trials run, {len(trials)} completed, "
          f"{summ['elapsed_sec'] / 3600:.1f} h | best trial {summ['best_trial']}: "
          f"val Gini {summ['best_val_gini']:.4f}, holdout Gini {summ['best_holdout_gini']:.4f}")
    t = trials.sort_values('val_gini', ascending=False).head(top)
    print(t.to_string(index=False, formatters={'val_gini': '{:.4f}'.format, 'holdout_gini': '{:.4f}'.format,
                                               'val_last_note': '{:.4f}'.format, 'lr': '{:.2e}'.format,
                                               'dropout': '{:.3f}'.format}))
    bp = summ['best_params']
    print('best params:', {k: (round(v, 6) if isinstance(v, float) else v) for k, v in bp.items() if not k.startswith('dim_')})
    print('per-summary dims:', {g: bp[f'dim_{g}'] for g in GROUPS_SHORT})
    print()
    return trials


att_trials = show_trials('Attorney')
lit_trials = show_trials('Litigation')

Attorney (attorney): 50 trials run, 43 completed, 9.2 h | best trial 42: val Gini 0.4999, holdout Gini 0.4605
 trial val_gini holdout_gini val_last_note  best_epoch       lr dropout  batch  fuse_dim
    42   0.4999       0.4605        0.8275           4 3.44e-04   0.696    256       256
    29   0.4996       0.4586        0.8303           6 8.93e-04   0.642   2048       512
    35   0.4984       0.4611        0.8261           7 3.11e-04   0.695   2048       256
    43   0.4981       0.4634        0.8267           3 3.76e-04   0.692    256       256
    34   0.4978       0.4623        0.8196           6 3.08e-04   0.696   2048       256
    41   0.4976       0.4616        0.8203           6 3.07e-04   0.699   2048       256
    31   0.4967       0.4619        0.8206           5 2.93e-04   0.655   2048       384
    30   0.4961       0.4601        0.8288           3 2.75e-04   0.662   2048       384
    47   0.4951       0.4568        0.8183           2 3.92e-04   0.595    256       256


### Step 4c — rebuild the winning model for each target from its best trial's settings

In [ ]:
def cfg_from_best(study):
    bp = load_json(f'{RUN_PREFIX}/{study}/_SUMMARY.json')['best_params']  # S3 read
    return {'lr': bp['lr'], 'dropout': bp['dropout'], 'weight_decay': bp['weight_decay'],
            'noise_std': bp['noise_std'], 'batch': bp['batch'], 'fuse_dim': bp['fuse_dim'],
            'group_dims': [bp[f'dim_{g}'] for g in GROUPS]}


torch.manual_seed(42); np.random.seed(42)
att_net, att_val = train_one(cfg_from_best(STUDIES['Attorney']), *att_data['train'][:2], *att_data['val'][:2])
lit_net, lit_val = train_one(cfg_from_best(STUDIES['Litigation']), *lit_data['train'][:2], *lit_data['val'][:2])

## Step 5 — best models on train / val / holdout

In [9]:
def report(name, net, data):
    print(f'{name:<11}' + ''.join(f'{s:>10}' for s in ['train', 'val', 'holdout']))
    for level in ['row-level', 'last-snapshot']:
        vals = []
        for s in ['train', 'val', 'holdout']:
            X, y, claims, _ = data[s]
            p = predict(net, X)
            if level == 'last-snapshot':                # each coverage's latest snapshot
                last = pd.Series(np.arange(len(claims))).groupby(claims).last().to_numpy()
                p, y = p[last], y[last]
            vals.append(gini(y, p))
        print(f'  {level + " Gini":<16}' + ''.join(f'{v:>9.3f}' for v in vals))

report('Attorney', att_net, att_data)
report('Litigation', lit_net, lit_data)

Attorney        train       val   holdout
  row-level Gini      0.667    0.500    0.461
  last-snapshot Gini     0.892    0.828    0.818
Litigation      train       val   holdout
  row-level Gini      0.636    0.449    0.465
  last-snapshot Gini     0.858    0.780    0.762


## Step 6 — holdout Gini at each cutoff day

In [10]:
DAY_GRIDS = {'Attorney': [1, 7, 15, 30, 60, 90, 180, 365, 730, 1095],
             'Litigation': [1, 90, 180, 270, 365, 540, 730, 810, 900, 1095, 1460]}


def gini_by_day(name, net, data):
    X, y, cvg, day = data['holdout']
    df = pd.DataFrame({'day': day, 'p': predict(net, X), 'y': y})
    print(name); print(f"  {'day':>5} {'claims':>7} {'with event':>11} {'Gini':>6}")
    for d in DAY_GRIDS[name]:
        snap = df[df.day == d]
        print(f'  {d:>5} {len(snap):>7,} {int(snap.y.sum()):>11,} {gini(snap.y, snap.p):>6.3f}')

gini_by_day('Attorney', att_net, att_data)
gini_by_day('Litigation', lit_net, lit_data)

Attorney
    day  claims  with event   Gini
      1   9,015       3,828  0.313
      7   8,030       3,229  0.337
     15   7,459       2,716  0.364
     30   6,571       1,968  0.392
     60   5,475       1,241  0.424
     90   4,818         956  0.446
    180   3,350         577  0.443
    365   1,816         296  0.454
    730     819          98  0.413
   1095     429          20  0.542
Litigation
    day  claims  with event   Gini
      1   9,997       1,141  0.368
     90   8,232       1,066  0.504
    180   6,923       1,015  0.525
    270   5,903         936  0.546
    365   4,987         823  0.544
    540   3,709         614  0.553
    730   2,428         291  0.571
    810   1,962         161  0.484
    900   1,610          95  0.386
   1095   1,074          54  0.304
   1460     486          14  0.344
